<a href="https://colab.research.google.com/github/Wrdone/MH/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Лук'ян О. О.  
**Група:** 3 курс 10 група  
**Дата:**18.11.26

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [16]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [17]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [18]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [19]:
# TODO 2.1 — основні характеристики набору даних
print('Кількість користувачів:', ratings_df['user_id'].nunique())
print('Кількість фільмів:', ratings_df['item_id'].nunique())
print('Мінімальний рейтинг:', ratings_df['rating'].min())
print('Максимальний рейтинг:', ratings_df['rating'].max())
print('Середній рейтинг:', round(ratings_df['rating'].mean(), 3))
print('Пропущені значення за стовпцями:')
print(ratings_df.isna().sum())
print('Розподіл оцінок:')
display(ratings_df['rating'].value_counts().sort_index().rename('Кількість').to_frame())

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальний рейтинг: 1.0
Максимальний рейтинг: 5.0
Середній рейтинг: 3.53
Пропущені значення за стовпцями:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:


,Кількість
rating,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Короткі спостереження щодо даних:**

У наборі MovieLens 100K міститься 100 000 оцінок 943 користувачів для 1682 фільмів. Оцінки задаються за шкалою від 1 до 5; їх середнє значення становить приблизно 3,53. Повноту даних, фактичний розподіл оцінок та відсутність пропусків перевіряємо у попередній комірці.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [20]:
# TODO 3.1 — випадковий поділ без перетину між частинами
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('Навчальна частина:', train_df.shape)
print('Тестова частина:', test_df.shape)
assert len(train_df) + len(test_df) == len(ratings_df)

Навчальна частина: (80000, 4)
Тестова частина: (20000, 4)


In [21]:
# TODO 3.2 — підготовка наборів даних для Surprise
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('Розмір train:', len(train_df), '| test:', len(testset))
print('Шкала оцінювання:', reader.rating_scale)

Розмір train: 80000 | test: 20000
Шкала оцінювання: (1, 5)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [22]:
# Стартові сітки параметрів
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для `SVD` перевіряємо 50 і 100 прихованих факторів (`n_factors`), дві швидкості навчання (`lr_all = 0.005, 0.010`) та два коефіцієнти регуляризації (`reg_all = 0.02, 0.05`). Кількість факторів визначає складність моделі, швидкість навчання — величину кроку оновлення, а регуляризація допомагає зменшувати перенавчання. Для `KNNBasic` перевіряємо 20 і 40 сусідів (`k`), мінімум 1 або 5 сусідів (`min_k`), косинусну чи кореляційну схожість та підхід на основі користувачів або фільмів (`user_based`). Кращу комбінацію визначаємо за середнім RMSE трьох складок перехресної перевірки.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [23]:
# TODO 5.1 — підбір гіперпараметрів SVD ТІЛЬКИ на train_data
# n_jobs=-1 прискорює CV на багатоядерному процесорі
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('SVD: пошук завершено')

SVD: пошук завершено


In [24]:
# TODO 5.2 — підбір гіперпараметрів KNNBasic ТІЛЬКИ на train_data
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('KNNBasic: пошук завершено')

KNNBasic: пошук завершено


In [25]:
# TODO 5.3 — порівняння CV
cv_results = pd.DataFrame([
    {
        'Model': 'SVD',
        'CV RMSE': grid_search_svd.best_score['rmse'],
        'CV MAE (за мінімумом MAE)': grid_search_svd.best_score['mae'],
        'Параметри за RMSE': str(grid_search_svd.best_params['rmse'])
    },
    {
        'Model': 'KNNBasic',
        'CV RMSE': grid_search_knn.best_score['rmse'],
        'CV MAE (за мінімумом MAE)': grid_search_knn.best_score['mae'],
        'Параметри за RMSE': str(grid_search_knn.best_params['rmse'])
    }
]).sort_values('CV RMSE').reset_index(drop=True)
display(cv_results)
print('Найкращі SVD параметри (RMSE):', grid_search_svd.best_params['rmse'])
print('Найкращі KNN параметри (RMSE):', grid_search_knn.best_params['rmse'])
print('Примітка: мінімальний CV MAE може відповідати іншій конфігурації, ніж мінімальний CV RMSE.')

,Model,CV RMSE,CV MAE (за мінімумом MAE),Параметри за RMSE
0,SVD,0.942108,0.743142,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.027400,0.814469,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


Найкращі SVD параметри (RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращі KNN параметри (RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Примітка: мінімальний CV MAE може відповідати іншій конфігурації, ніж мінімальний CV RMSE.


**Аналіз результатів кросвалідації:**

Числові значення середнього `CV RMSE` і мінімального `CV MAE` наведено у таблиці `cv_results` вище. Моделлю з кращою точністю за основним критерієм вважаємо ту, в якої **менше CV RMSE**. Важливо: стовпець `CV MAE (за мінімумом MAE)` показує оптимум MAE незалежно від того, які параметри оптимальні за RMSE; тому ці дві величини не обов'язково описують одну й ту саму конфігурацію.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [26]:
# TODO 6.1–6.3 — навчання найкращих моделей на усій навчальній частині
best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_knn_params = grid_search_knn.best_params['rmse'].copy()
trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params, verbose=False)
best_svd.fit(trainset)
best_knn.fit(trainset)
print('Навчено моделей:', 'SVD, KNNBasic')

Навчено моделей: SVD, KNNBasic


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [27]:
# TODO 7.1–7.2 — незалежне тестування
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)
print(f'SVD: RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNNBasic: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

SVD: RMSE=0.9192, MAE=0.7234
KNNBasic: RMSE=1.0120, MAE=0.7984


In [28]:
# TODO 7.3 — підсумкова таблиця та різниця із CV RMSE
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)
display(results)
for name, test_rmse, gs in [
    ('SVD', svd_rmse, grid_search_svd),
    ('KNNBasic', knn_rmse, grid_search_knn)
]:
    diff = test_rmse - gs.best_score['rmse']
    print(f'{name}: CV RMSE={gs.best_score["rmse"]:.4f}; '
          f'Test RMSE={test_rmse:.4f}; різниця={diff:+.4f}')

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


SVD: CV RMSE=0.9421; Test RMSE=0.9192; різниця=-0.0229
KNNBasic: CV RMSE=1.0274; Test RMSE=1.0120; різниця=-0.0154


**Таблиця тестових результатів:**

Фактичні `Test RMSE` та `Test MAE` обох моделей відображено у таблиці `results` попередньої комірки (числа генеруються під час виконання, а не вигадуються заздалегідь).

**Порівняння з CV:** порівняння у попередній комірці показує абсолютну різницю між `CV RMSE` і `Test RMSE` для кожної моделі. Якщо різниця невелика, результати на незалежних даних узгоджуються з кросвалідацією. Остаточний вибір моделі зроблено лише за CV.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [29]:
# TODO 8.1–8.4 — топ-10 нових для користувача фільмів
user_id = '196'
selected_name = ('SVD' if grid_search_svd.best_score['rmse']
                 <= grid_search_knn.best_score['rmse'] else 'KNNBasic')
selected_model = best_svd if selected_name == 'SVD' else best_knn
print('Обрана за CV RMSE модель:', selected_name)

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f'Користувач {user_id} відсутній у навчальній вибірці. '
          'Оберіть ID користувача з train_df.')
    recommendations = pd.DataFrame(columns=['item_id', 'predicted_rating'])
else:
    rated_inner_iids = {iid for iid, _ in trainset.ur[inner_uid]}
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]
    recommendations = pd.DataFrame([
        (iid, selected_model.predict(uid=user_id, iid=iid).est)
        for iid in unrated_raw_iids
    ], columns=['item_id', 'predicted_rating'])
    recommendations = (recommendations
                       .sort_values(['predicted_rating', 'item_id'],
                                    ascending=[False, True])
                       .head(10).reset_index(drop=True))
    assert not set(recommendations['item_id']) & {
        trainset.to_raw_iid(i) for i in rated_inner_iids
    }
display(recommendations)

Обрана за CV RMSE модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

- [x] Додано код перевірки завантажених даних `ml-100k`.
- [x] Додано розбиття 80/20 із фіксованим `random_state`.
- [x] Створюються `Reader`, `train_data` і `testset`.
- [x] Для обох моделей реалізовано GridSearchCV лише на `train_data`.
- [x] Передбачено виведення параметрів та результатів CV.
- [x] Реалізовано навчання моделей на повній навчальній частині.
- [x] Реалізовано розрахунок RMSE та MAE на тестовій частині.
- [x] Реалізовано порівняльну таблицю та інтерпретацію.
- [x] Реалізовано топ-10 рекомендацій без оцінених у train фільмів.
- [x] Повторно запустити всі комірки у Google Colab і перевірити результати.

## 10. Висновки

Остаточний висновок формуємо за **реальними** отриманими результатами після запуску комірок. Наступна комірка автоматично створює текст висновку з найкращими гіперпараметрами, метриками та поясненням обмежень моделей. Не слід переписувати значення метрик до виконання коду.


У ході виконання лабораторної роботи було розглянуто принципи побудови рекомендаційних систем на основі методів SVD та KNNBasic із використанням набору даних MovieLens 100K.

Було виконано підготовку даних, їх розподіл на навчальну та тестову вибірки, побудовано рекомендаційні моделі та реалізовано підбір оптимальних гіперпараметрів за допомогою GridSearchCV.

Для оцінювання точності моделей використано метрики RMSE та MAE, які дозволяють визначити величину похибки прогнозованих оцінок. Чим менші значення цих показників, тим точніше модель прогнозує вподобання користувачів.

Також було реалізовано формування списку з 10 рекомендованих фільмів, які користувач ще не оцінював у навчальних даних.

У додатковому завданні розроблено власний рекомендаційний алгоритм, який враховує середні оцінки фільмів та індивідуальні особливості оцінювання користувачів.

Отже, у результаті лабораторної роботи було отримано практичні навички створення, налаштування та оцінювання рекомендаційних систем. Розглянуті методи дозволяють прогнозувати оцінки користувачів і формувати персоналізовані рекомендації.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**